## import libraries

In [37]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
from shapely.geometry import MultiPolygon
import warnings

## read datasets

In [38]:
data_root = Path("_data/eurocrops2")

In [39]:
def get_inventory(root_path):
    inventory = []
    
    for country_dir in root_path.iterdir():
        if country_dir.is_dir():
            for file_path in country_dir.glob("*.parquet"):
                parts = file_path.stem.split('_')
                
                country_id = parts[0]   
                year = int(parts[-1])   
                country = country_dir.name  
                
                inventory.append({"country": country, 
                                  "sub_region": country_id, 
                                  "year": year, 
                                  "id": f"{country}_{year}", 
                                  "sub_region_id": f"{country_id}_{year}", 
                                  "path": file_path})
    
    df = pd.DataFrame(inventory)
    
    if not df.empty:
        df = df.sort_values(by=["country", "year"]).reset_index(drop=True)
        
    return df

In [40]:
available_datasets = get_inventory(data_root)
available_datasets

,country,sub_region,year,id,sub_region_id,path
0,at,at,2016,at_2016,at_2016,_data/eurocrops2/at/at_2016.parquet
1,at,at,2018,at_2018,at_2018,_data/eurocrops2/at/at_2018.parquet
2,at,at,2019,at_2019,at_2019,_data/eurocrops2/at/at_2019.parquet
3,be,be2,2016,be_2016,be2_2016,_data/eurocrops2/be/be2_2016.parquet
4,be,be3,2016,be_2016,be3_2016,_data/eurocrops2/be/be3_2016.parquet
5,be,be2,2018,be_2018,be2_2018,_data/eurocrops2/be/be2_2018.parquet
6,be,be3,2018,be_2018,be3_2018,_data/eurocrops2/be/be3_2018.parquet
7,be,be3,2019,be_2019,be3_2019,_data/eurocrops2/be/be3_2019.parquet
8,be,be2,2019,be_2019,be2_2019,_data/eurocrops2/be/be2_2019.parquet
9,bg,bg,2016,bg_2016,bg_2016,_data/eurocrops2/bg/bg_2016.parquet


## select columns of interest

In [5]:
# see what columns we have on the datasets
gpd.read_parquet(available_datasets['path'][0]).head(3)

,cropfield,original_code,off_id,area_ha,geometry
0,650256,111,30987880,0.660696,"MULTIPOLYGON (((2752431.766 1740785.231, 27524..."
1,653074,111,2988311,1.213365,"MULTIPOLYGON (((2754679.823 1755722.8, 2754685..."
2,1687121,10036,37116587,0.167639,"MULTIPOLYGON (((2766743.279 2014071.901, 27667..."


In [6]:
columns = ['original_code', 'geometry']

In [7]:
def load_and_filter_datasets(dataframe, columns_of_interest):
    processed = {}
    
    for _, row in dataframe.iterrows():
        gdf = gpd.read_parquet(row['path'])
        processed[row['sub_region_id']] = gdf[columns_of_interest].copy()
        
    return processed

In [8]:
datasets = load_and_filter_datasets(available_datasets, columns)
datasets.keys()

dict_keys(['pt_2017', 'pt_2018', 'pt_2019'])

In [9]:
datasets['pt_2019'].head(2)

,original_code,geometry
0,89,"MULTIPOLYGON (((2900991.807 2205061.434, 29009..."
1,170,"MULTIPOLYGON (((2726284.525 1953915.285, 27262..."


## mapping the crops on the dataset

In [10]:
target_cols = ['original_code', 'geometry', 'hcat4_code', 'hcat4_name']

In [11]:
def load_and_map_datasets(inventory_df, mapping_csv_path, columns_of_interest):
    mapping_df = pd.read_csv(mapping_csv_path)

    mapping_df["original_code"] = mapping_df["original_code"].astype(str).str.strip()
    mapping_df["nuts"] = mapping_df["nuts"].astype(str).str.strip()

    processed = {}

    for _, row in inventory_df.iterrows():
        print(f"Processing {row['sub_region_id']}")
        gdf = gpd.read_parquet(row["path"])

        gdf["original_code"] = gdf["original_code"].astype(str).str.strip()

        country_map = mapping_df[mapping_df["nuts"] == row["sub_region"]]
        if country_map.empty:
            country_map = mapping_df[mapping_df["nuts"] == row["country"]]

        merged_gdf = gdf.merge(country_map[["original_code", 
                                            "hcat4_code", 
                                            "hcat4_name"]], 
                                            on="original_code", 
                                            how="left")
        
        final_cols = [c for c in columns_of_interest if c in merged_gdf.columns]
        processed[row["sub_region_id"]] = merged_gdf[final_cols].copy()
        
    return processed

In [12]:
datasets['pt_2018']

,original_code,geometry
0,143,"MULTIPOLYGON (((2743757.997 1916390.235, 27437..."
1,143,"MULTIPOLYGON (((2753096.453 1909340.83, 275310..."
2,143,"MULTIPOLYGON (((2744108.547 1919800.115, 27441..."
3,143,"MULTIPOLYGON (((2754315.715 1908925.977, 27543..."
4,143,"MULTIPOLYGON (((2753881.3 1908862.659, 2753846..."
...,...,...
3024829,10233,"MULTIPOLYGON (((2934285.756 2240332.899, 29342..."
3024830,10233,"MULTIPOLYGON (((2871376.034 2147967.104, 28713..."
3024831,10233,"MULTIPOLYGON (((2961897.25 2191106.541, 296189..."
3024832,10233,"MULTIPOLYGON (((2961594.238 2191808.436, 29615..."


In [13]:
datasets = load_and_map_datasets(available_datasets, "_data/eurocrops2_original_data/eurocrops.csv", target_cols)
datasets.keys()

Processing pt_2017
Processing pt_2018
Processing pt_2019


dict_keys(['pt_2017', 'pt_2018', 'pt_2019'])

In [14]:
datasets['pt_2019'].head(3)

,original_code,geometry,hcat4_code,hcat4_name
0,89,"MULTIPOLYGON (((2900991.807 2205061.434, 29009...",3311100000,fallow_land_not_crop
1,170,"MULTIPOLYGON (((2726284.525 1953915.285, 27262...",3360000000,tree_wood_forest
2,262,"MULTIPOLYGON (((2702764.982 1837437.864, 27028...",3360600000,oak


## eliminate Nan values on geometry

In [15]:
def eliminate_Nan(datasets):
    cleaned_datasets = {}
    
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
        
        for key in datasets.keys():
            gdf = datasets[key]
            
            missing_count = gdf.geometry.isna().sum()
            empty_count = gdf.geometry.is_empty.sum()
            
            if missing_count > 0 or empty_count > 0:
                print(f"Cleaning '{key}': Removed {missing_count} NaN and {empty_count} Empty.")
            
            cleaned_gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
            cleaned_datasets[key] = cleaned_gdf
            
    return cleaned_datasets

In [16]:
datasets = eliminate_Nan(datasets)

Cleaning 'pt_2017': Removed 0 NaN and 52 Empty.
Cleaning 'pt_2018': Removed 0 NaN and 53 Empty.
Cleaning 'pt_2019': Removed 0 NaN and 108 Empty.


## select crops of interest

maize_corn_popcorn 

winter_barley

potatoes

In [17]:
# # count the most occurent classes
# print(f"Most common classes: {datasets["pt_2018"]['hcat4_name'].value_counts()[:25]} \n")

In [18]:
# define crops of interest
crops = ['winter_barley', 'potatoes', 'maize_corn_popcorn']

def interest_crop(gdf, crop):
    result = gdf[gdf['hcat4_name'] == crop]
    return result, len(result)

crops_dataset = {}

for dataset_id, gdf in datasets.items():

    for crop in crops:

        var_name = f"{crop}_{dataset_id}"

        subset_gdf, subset_size = interest_crop(gdf, crop)

        if subset_size > 0:
            crops_dataset[var_name] = subset_gdf

In [19]:
print(f"Old dataset keys: {datasets.keys()}")
print(f"New dataset keys: {crops_dataset.keys()}")
print(f"Number of new datasets created: {len(crops_dataset)}")
print(f"New datasets created: {crops_dataset.keys()}")

Old dataset keys: dict_keys(['pt_2017', 'pt_2018', 'pt_2019'])
New dataset keys: dict_keys(['potatoes_pt_2017', 'maize_corn_popcorn_pt_2017', 'potatoes_pt_2018', 'maize_corn_popcorn_pt_2018', 'potatoes_pt_2019', 'maize_corn_popcorn_pt_2019'])
Number of new datasets created: 6
New datasets created: dict_keys(['potatoes_pt_2017', 'maize_corn_popcorn_pt_2017', 'potatoes_pt_2018', 'maize_corn_popcorn_pt_2018', 'potatoes_pt_2019', 'maize_corn_popcorn_pt_2019'])


## merging back sub regions and add year and country id

In [20]:
target_countries = available_datasets['country'].unique()
target_countries

array(['pt'], dtype=object)

In [21]:
def merge_sub_regions(dataset_dict, countries=['be']):
    if isinstance(countries, str):
        countries = [countries]
        
    merged_results = {}

    for country in countries:
        pattern = rf'_{country}\d+_' 
        standard_pattern = rf'_{country}_'
        
        sub_region_keys = [k for k in dataset_dict.keys() if re.search(pattern, k)]
        
        if not sub_region_keys:
            standard_keys = [k for k in dataset_dict.keys() if re.search(standard_pattern, k)]
            
            for k in standard_keys:
                parts = k.split('_')
                df_temp = dataset_dict[k].copy()
                df_temp['country_id'] = country
                df_temp['year'] = parts[-1]
                merged_results[k] = df_temp
            continue 

        groups = {}
        for k in sub_region_keys:
            new_key = re.sub(pattern, f'_{country}_', k)
            
            parts = k.split('_')
            year = parts[-1]        
            country_id = country    
            
            df_temp = dataset_dict[k].copy()
            df_temp['country_id'] = country_id
            df_temp['year'] = year
            
            if new_key not in groups:
                groups[new_key] = []
            groups[new_key].append(df_temp)
            
        for target_name, df_list in groups.items():
            merged_results[target_name] = pd.concat(df_list, ignore_index=True)
            
    return merged_results

In [22]:
crops_dataset = merge_sub_regions(crops_dataset, countries=target_countries)
crops_dataset.keys()

dict_keys(['potatoes_pt_2017', 'maize_corn_popcorn_pt_2017', 'potatoes_pt_2018', 'maize_corn_popcorn_pt_2018', 'potatoes_pt_2019', 'maize_corn_popcorn_pt_2019'])

In [23]:
crops_dataset['maize_corn_popcorn_pt_2019'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,country_id,year
7867,6,"MULTIPOLYGON (((2778592.531 2229659.43, 277860...",3310106000,maize_corn_popcorn,pt,2019
18220,6,"MULTIPOLYGON (((2767505.739 2220886.243, 27674...",3310106000,maize_corn_popcorn,pt,2019
20557,6,"MULTIPOLYGON (((2782378.278 2261194.068, 27823...",3310106000,maize_corn_popcorn,pt,2019


## eliminate multipoligons with more than one poligon

In [24]:
def drop_multipoligons(datasets):
    cleaned_results = {}
    
    for name, gdf in datasets.items():
        multi_parts_indices = []
        eliminates = 0
        
        for idx, geom in gdf['geometry'].items():
            if isinstance(geom, MultiPolygon) and len(geom.geoms) > 1:
                multi_parts_indices.append(idx)
                eliminates = eliminates + 1
        
        print(f"Eliminating multipoligon {eliminates} poligons in '{name}'.")
        
        if multi_parts_indices:
            cleaned_gdf = gdf.drop(index=multi_parts_indices)
            cleaned_results[name] = cleaned_gdf.reset_index(drop=True)
        else:
            cleaned_results[name] = gdf.copy()
            
    return cleaned_results

In [25]:
crops_dataset = drop_multipoligons(crops_dataset)

Eliminating multipoligon 0 poligons in 'potatoes_pt_2017'.
Eliminating multipoligon 2 poligons in 'maize_corn_popcorn_pt_2017'.
Eliminating multipoligon 0 poligons in 'potatoes_pt_2018'.
Eliminating multipoligon 1 poligons in 'maize_corn_popcorn_pt_2018'.
Eliminating multipoligon 0 poligons in 'potatoes_pt_2019'.
Eliminating multipoligon 7 poligons in 'maize_corn_popcorn_pt_2019'.


In [26]:
crops_dataset['maize_corn_popcorn_pt_2019']

,original_code,geometry,hcat4_code,hcat4_name,country_id,year
0,6,"MULTIPOLYGON (((2778592.531 2229659.43, 277860...",3310106000,maize_corn_popcorn,pt,2019
1,6,"MULTIPOLYGON (((2767505.739 2220886.243, 27674...",3310106000,maize_corn_popcorn,pt,2019
2,6,"MULTIPOLYGON (((2782378.278 2261194.068, 27823...",3310106000,maize_corn_popcorn,pt,2019
3,6,"MULTIPOLYGON (((2782422.871 2261174.547, 27824...",3310106000,maize_corn_popcorn,pt,2019
4,6,"MULTIPOLYGON (((2847625.14 2060882.743, 284762...",3310106000,maize_corn_popcorn,pt,2019
...,...,...,...,...,...,...
167695,6,"MULTIPOLYGON (((2950538.194 2214362.242, 29505...",3310106000,maize_corn_popcorn,pt,2019
167696,6,"MULTIPOLYGON (((2926530.437 2180253.701, 29265...",3310106000,maize_corn_popcorn,pt,2019
167697,6,"MULTIPOLYGON (((2908003.212 2240664.634, 29080...",3310106000,maize_corn_popcorn,pt,2019
167698,6,"MULTIPOLYGON (((2917189.28 2244232.814, 291718...",3310106000,maize_corn_popcorn,pt,2019


## limit datasets to a certain number of rows

In [27]:
data_size = 500

In [28]:
crops_dataset = {name: df.iloc[:data_size] for name, df in crops_dataset.items()}

## get centroid and long and lat

In [29]:
def access_coords(gpd, index):
    poligon = gpd.geometry.iloc[index]

    coords = list(poligon.exterior.coords)

    coords_numpy = np.array(poligon.exterior.coords)
    return poligon, coords, coords_numpy

In [30]:
def get_centroid(gdf, centroid_as_main=False):
    gdf = gdf.copy()
    local_utm = gdf.estimate_utm_crs()
    gdf['centroid'] = gdf.geometry.to_crs(local_utm).centroid.to_crs(epsg=4326)
    gdf = gdf.to_crs(epsg=4326)
    
    if centroid_as_main: 
        gdf = gdf.set_geometry('centroid')
        
    return gdf

In [31]:
def get_long_lat(gdf):
    gdf['long'] = gdf['centroid'].x
    gdf['lat'] = gdf['centroid'].y
    return gdf

In [32]:
crops_dataset['maize_corn_popcorn_pt_2019'].head(3)

,original_code,geometry,hcat4_code,hcat4_name,country_id,year
0,6,"MULTIPOLYGON (((2778592.531 2229659.43, 277860...",3310106000,maize_corn_popcorn,pt,2019
1,6,"MULTIPOLYGON (((2767505.739 2220886.243, 27674...",3310106000,maize_corn_popcorn,pt,2019
2,6,"MULTIPOLYGON (((2782378.278 2261194.068, 27823...",3310106000,maize_corn_popcorn,pt,2019


In [33]:
# check before applying transformations
print(crops_dataset['maize_corn_popcorn_pt_2019'].crs)
crops_dataset['maize_corn_popcorn_pt_2019'].head(3)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89-extended / LAEA Europe", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}, "accuracy": "0.1", "id": {"authority":

,original_code,geometry,hcat4_code,hcat4_name,country_id,year
0,6,"MULTIPOLYGON (((2778592.531 2229659.43, 277860...",3310106000,maize_corn_popcorn,pt,2019
1,6,"MULTIPOLYGON (((2767505.739 2220886.243, 27674...",3310106000,maize_corn_popcorn,pt,2019
2,6,"MULTIPOLYGON (((2782378.278 2261194.068, 27823...",3310106000,maize_corn_popcorn,pt,2019


In [34]:
for i in crops_dataset:
    print(f"Apply functions on {i}")
    crops_dataset[i] = get_centroid(crops_dataset[i])
    crops_dataset[i] = get_long_lat(crops_dataset[i])

Apply functions on potatoes_pt_2017
Apply functions on maize_corn_popcorn_pt_2017
Apply functions on potatoes_pt_2018
Apply functions on maize_corn_popcorn_pt_2018
Apply functions on potatoes_pt_2019
Apply functions on maize_corn_popcorn_pt_2019


In [35]:
# check after applying transformations
print(crops_dataset['maize_corn_popcorn_pt_2019'].crs)
crops_dataset['maize_corn_popcorn_pt_2019']

EPSG:4326


,original_code,geometry,hcat4_code,hcat4_name,country_id,year,centroid,long,lat
0,6,"MULTIPOLYGON (((-8.59542 41.44688, -8.59534 41...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.59517 41.44699),-8.595169,41.446993
1,6,"MULTIPOLYGON (((-8.70083 41.34572, -8.70113 41...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.69982 41.34551),-8.699815,41.345510
2,6,"MULTIPOLYGON (((-8.63676 41.73051, -8.6367 41....",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.63677 41.73047),-8.636767,41.730471
3,6,"MULTIPOLYGON (((-8.63618 41.73044, -8.63639 41...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.63625 41.73048),-8.636249,41.730480
4,6,"MULTIPOLYGON (((-7.36491 40.11598, -7.36493 40...",3310106000,maize_corn_popcorn,pt,2019,POINT (-7.36509 40.11639),-7.365094,40.116387
...,...,...,...,...,...,...,...,...,...
495,6,"MULTIPOLYGON (((-8.22111 40.53333, -8.22112 40...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.22127 40.53352),-8.221273,40.533523
496,6,"MULTIPOLYGON (((-8.22267 40.53665, -8.22215 40...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.22241 40.53657),-8.222415,40.536575
497,6,"MULTIPOLYGON (((-8.21619 40.53692, -8.21632 40...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.21598 40.53716),-8.215985,40.537157
498,6,"MULTIPOLYGON (((-8.28032 40.92497, -8.28051 40...",3310106000,maize_corn_popcorn,pt,2019,POINT (-8.2805 40.9253),-8.280502,40.925301


## export data

In [36]:
export_folder = "_data/exports/crop_country_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for name in crops_dataset.keys():
    temp_df = pd.DataFrame(crops_dataset[name][['country_id', 'year', 'hcat4_code', 
                                                'hcat4_name', 'long', 'lat']]).copy()
    
    file_path = f"{export_folder}/{name}.csv"
    
    temp_df.to_csv(file_path, index=False)
    print(f"Exported: {file_path}")

Exported: _data/crop_country_exports/potatoes_pt_2017.csv
Exported: _data/crop_country_exports/maize_corn_popcorn_pt_2017.csv
Exported: _data/crop_country_exports/potatoes_pt_2018.csv
Exported: _data/crop_country_exports/maize_corn_popcorn_pt_2018.csv
Exported: _data/crop_country_exports/potatoes_pt_2019.csv
Exported: _data/crop_country_exports/maize_corn_popcorn_pt_2019.csv
